# SI4006 · Sesión 12 · Lab: **Del ruido a la imagen — difusión, guidance y ControlNet**

**Tópicos Especiales y Aplicaciones en IA** · Universidad EAFIT · Módulo 4

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

---

En la S11 la imagen era la **entrada**. Hoy es la **salida**: generamos imágenes con un modelo de difusión y aprendemos a medirlas y a controlarlas. Dos labs:

- **Lab A · La difusión por dentro + guidance (≈25 min):** ensuciar una imagen con ruido (forward), ver cómo aparece una imagen desde el ruido (reverse), y medir qué hace el `guidance_scale` con CLIP score y cronómetro.
- **Lab B · Control con ControlNet (≈25 min):** fijar la composición con los bordes de una imagen de referencia y generar para SU dominio.

> **Traigan 1–2 imágenes de SU dominio** y una idea de prompt. SD 1.5 entiende **inglés**: escriban los prompts en inglés.
>
> Corre en Colab con GPU **T4** (Entorno de ejecución → Cambiar tipo → T4). En CPU la difusión es impracticable (minutos por imagen).

## 0 · Setup

In [ ]:
# Colab ya trae torch, transformers y opencv. Actualizamos diffusers (Colab trae una versión vieja).
%pip install -q -U diffusers "transformers>=4.45" accelerate
print('Listo. Si Colab pide reiniciar la sesión, reinicien y sigan desde la celda siguiente.')

In [ ]:
import torch, diffusers, transformers, time, gc, io, requests, inspect
import numpy as np, cv2
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from packaging import version

device = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if device == 'cuda' else torch.float32
# diffusers >= 0.40 renombró torch_dtype → dtype (el viejo sigue, con aviso). Usamos el que toque.
DT = {'dtype': DTYPE} if version.parse(diffusers.__version__) >= version.parse('0.40.0') else {'torch_dtype': DTYPE}

print('diffusers', diffusers.__version__, '| transformers', transformers.__version__, '| torch', torch.__version__)
print('device:', device, '| dtype:', DTYPE)
if device == 'cpu':
    print('\n⚠️  Sin GPU. Generar una imagen en CPU toma varios minutos: cambien a T4 antes de seguir.')

In [ ]:
def cargar(src):
    if src.startswith('http'):
        return Image.open(io.BytesIO(requests.get(src, timeout=30).content)).convert('RGB')
    return Image.open(src).convert('RGB')

def cuadrada(img, lado=512):
    # recorte centrado a lado×lado (SD 1.5 fue entrenado a 512×512)
    return ImageOps.fit(img, (lado, lado), Image.LANCZOS)

def mostrar(imagenes, titulos=None, tam=3.2):
    n = len(imagenes)
    fig, axs = plt.subplots(1, n, figsize=(tam * n, tam))
    axs = np.atleast_1d(axs)
    for k, (ax, im) in enumerate(zip(axs, imagenes)):
        ax.imshow(im); ax.axis('off')
        if titulos: ax.set_title(titulos[k], fontsize=10)
    plt.tight_layout(); plt.show()

def liberar():
    gc.collect()
    if device == 'cuda': torch.cuda.empty_cache()

print('Helpers listos.')

## 1 · El modelo base: Stable Diffusion 1.5

Stable Diffusion no es una sola red, es un equipo de cuatro piezas:

| Pieza | Qué hace |
|---|---|
| `text_encoder` + `tokenizer` | el encoder de texto de **CLIP** (el de la S11): el prompt → vectores |
| `unet` | la red que **predice el ruido** en cada paso, mirando el texto por cross-attention |
| `vae` | comprime la imagen 512×512×3 a un **latente** 4×64×64 y la reconstruye al final |
| `scheduler` | la receta de cuánto ruido quitar en cada paso |

Lo cargamos **una sola vez** en fp16 (~2,7 GB de descarga la primera vez) y reusamos sus piezas en todo el notebook. El `safety_checker` queda como viene: si marca una imagen, la devuelve negra.

In [ ]:
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
SD_ID = 'stable-diffusion-v1-5/stable-diffusion-v1-5'   # espejo comunitario (el repo de runwayml ya no existe)
pipe = StableDiffusionPipeline.from_pretrained(SD_ID, variant='fp16', **DT).to(device)
# El scheduler de fábrica (PNDM) necesita ~50 pasos; DPM-Solver++ da lo mismo en ~25.
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.set_progress_bar_config(leave=False)

for nombre, m in pipe.components.items():
    n = sum(p.numel() for p in m.parameters()) / 1e6 if isinstance(m, torch.nn.Module) else 0
    print(f'{nombre:18s} {type(m).__name__:32s} {n:8.0f} M' if n else f'{nombre:18s} {type(m).__name__}')

In [ ]:
SEED = 42   # misma semilla = mismo ruido inicial = comparación justa

def semilla(s=SEED):
    return torch.Generator(device).manual_seed(s)

def generar_img(p, prompt, seed=SEED, **kw):
    # genera UNA imagen 512×512 con semilla fija y devuelve (imagen, segundos)
    kw.setdefault('num_inference_steps', 25)
    if device == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    img = p(prompt, height=512, width=512, generator=semilla(seed), **kw).images[0]
    if device == 'cuda': torch.cuda.synchronize()
    return img, time.perf_counter() - t0

print('generar_img lista.')

## 2 · Lab A · La difusión por dentro + guidance

### A1 · Forward process: ensuciar una imagen

El proceso hacia adelante no se aprende, es una fórmula. En el paso $t$:

$$x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, I)$$

$\bar\alpha_t$ baja de ~1 (t = 0) a casi 0 (t = 999): cada vez queda menos imagen y más ruido. `DDPMScheduler.add_noise(x0, ruido, t)` aplica exactamente esa fórmula.

Lo hacemos en **espacio de píxeles** para poder verlo. Stable Diffusion hace lo mismo, pero sobre el latente del VAE (A1b).

**TODO A1:** completen `ensuciar` con la llamada a `ddpm.add_noise`. Los pasos van como tensor de enteros: `torch.tensor([t])`.

> **Ojo:** esta celda falla hasta que completen el TODO (en la versión de clase devuelve `None`).

In [ ]:
from diffusers import DDPMScheduler
ddpm = DDPMScheduler.from_config(pipe.scheduler.config)   # misma tabla de ruido con la que se entrenó SD 1.5

# 👉 SU IMAGEN AQUÍ (URL o ruta). Por defecto, los gatos de COCO de la S11.
IMG_A = 'http://images.cocodataset.org/val2017/000000039769.jpg'
img_a = cuadrada(cargar(IMG_A))

def a_tensor(img):   # PIL [0,255] → tensor 1×3×512×512 en [-1, 1]
    return torch.from_numpy(np.array(img)).permute(2, 0, 1).float().unsqueeze(0) / 127.5 - 1

def a_pil(x):        # tensor [-1, 1] → PIL
    x = ((x.clamp(-1, 1) + 1) * 127.5).round().to(torch.uint8)
    return Image.fromarray(x[0].permute(1, 2, 0).cpu().numpy())

x0 = a_tensor(img_a)
ruido = torch.randn(x0.shape, generator=torch.Generator().manual_seed(0))

# TODO A1: devuelvan x0 con el ruido del paso t (forward process).
def ensuciar(x0, ruido, t):
    return None   # ← reemplacen

PASOS_T = [0, 100, 300, 600, 999]
fila = [a_pil(ensuciar(x0, ruido, t)) for t in PASOS_T]
mostrar(fila, [f't = {t}  ·  imagen {ddpm.alphas_cumprod[t].sqrt():.0%}' for t in PASOS_T])

> 👀 **Miren t = 999:** queda menos del 7 % de la imagen original; es casi ruido puro. Y ahora la idea clave: **entrenar un modelo de difusión = enseñarle a la U-Net a adivinar qué ruido $\varepsilon$ se le sumó**, dada la imagen sucia $x_t$, el paso $t$ y el texto. La pérdida es un simple MSE entre el ruido real y el predicho. Si sabe quitar un poco de ruido, lo aplicamos muchas veces y sale una imagen.

### A1b · ¿Por qué "latente"?

Hacer difusión sobre 512×512×3 píxeles es caro. Stable Diffusion la hace sobre el latente del VAE: 48 veces más pequeño.

In [ ]:
with torch.no_grad():
    xv = x0.to(device, dtype=pipe.vae.dtype)
    lat = pipe.vae.encode(xv).latent_dist.mean * pipe.vae.config.scaling_factor
    rec = pipe.vae.decode(lat / pipe.vae.config.scaling_factor).sample

print('imagen :', tuple(x0.shape), '→', x0[0].numel(), 'números')
print('latente:', tuple(lat.shape), '→', lat[0].numel(), f'números  ({x0[0].numel() / lat[0].numel():.0f}× más pequeño)')
canales = [(c - c.min()) / (c.max() - c.min()) for c in lat[0].float().cpu()]
mostrar([img_a, a_pil(rec.float())] + [c.numpy() for c in canales],
        ['original', 'VAE: codificar → decodificar'] + [f'latente canal {i}' for i in range(4)], tam=2.6)

### A2 · Reverse process: ver aparecer la imagen

Ahora al revés: partimos de ruido puro (fijado por la semilla) y la U-Net quita ruido durante 25 pasos. Para verlo, le pasamos al pipeline un `callback_on_step_end`: una función que diffusers llama al final de cada paso con la firma `(pipe, paso, timestep, callback_kwargs)` y que **debe devolver** `callback_kwargs`. Ahí vienen los `latents` del momento; guardamos una copia cada pocos pasos y al final los decodificamos con el VAE.

In [ ]:
# 👉 SU PROMPT AQUÍ (en inglés). Lo reusamos en todo el Lab A.
PROMPT = 'a cozy coffee shop in Medellin with plants and warm light, photograph'
N_PASOS = 25
capturas = []

def capturar(p, paso, t, kw):
    if paso % 4 == 0 or paso == N_PASOS - 1:
        capturas.append((paso, kw['latents'].detach().clone()))
    return kw   # obligatorio: el pipeline sigue con lo que devolvamos

img_final, seg = generar_img(pipe, PROMPT, num_inference_steps=N_PASOS,
                             callback_on_step_end=capturar, callback_on_step_end_tensor_inputs=['latents'])

@torch.no_grad()
def decodificar(lat):
    x = pipe.vae.decode(lat / pipe.vae.config.scaling_factor).sample
    return pipe.image_processor.postprocess(x, output_type='pil')[0]

cuadros = [decodificar(l) for _, l in capturas]
mostrar(cuadros, [f'paso {p + 1}/{N_PASOS}' for p, _ in capturas], tam=2.4)
print(f'{seg:.1f} s para {N_PASOS} pasos')

> 👀 En los primeros pasos solo aparecen manchas de color y la composición gruesa (dónde va lo claro, dónde lo oscuro); los detalles finos llegan al final. Primero estructura, luego textura: por eso ControlNet (Lab B) puede fijar la estructura sin quitarle libertad al texto.

### A3 · ¿Cuánto obedecer al texto? `guidance_scale`

En cada paso la U-Net predice el ruido dos veces: **con** el prompt y **sin** él. Classifier-free guidance (CFG) exagera la diferencia:

$$\hat\varepsilon = \varepsilon_{\text{sin texto}} + w\,(\varepsilon_{\text{con texto}} - \varepsilon_{\text{sin texto}})$$

Con $w = 1$ no hay guidance (y diffusers se salta la pasada sin texto: ojo al tiempo). Probamos $w$ = 1, 3, 7.5 y 15 con **el mismo prompt y la misma semilla**, y medimos la fidelidad al prompt con **CLIP score** = coseno(imagen, prompt) × 100, con el CLIP de la S11.

**TODO A3:** completen `clip_score`. `emb_imagen` y `emb_texto` ya devuelven vectores **normalizados**, así que el coseno es un producto punto.

> **Ojo:** esta celda falla hasta que completen el TODO (en la versión de clase devuelve `None`).

In [ ]:
from transformers import CLIPModel, CLIPProcessor
CLIP_ID = 'openai/clip-vit-base-patch32'
clip = CLIPModel.from_pretrained(CLIP_ID).to(device).eval()
proc = CLIPProcessor.from_pretrained(CLIP_ID)

def _vec(salida):
    # transformers 5 devuelve un objeto (el vector está en .pooler_output); la 4 devolvía el tensor directo
    return salida if isinstance(salida, torch.Tensor) else salida.pooler_output

@torch.no_grad()
def emb_imagen(img):
    ent = proc(images=[img], return_tensors='pt').to(device)
    return torch.nn.functional.normalize(_vec(clip.get_image_features(**ent)), dim=-1)

@torch.no_grad()
def emb_texto(texto):
    ent = proc(text=[texto], return_tensors='pt', padding=True, truncation=True).to(device)
    return torch.nn.functional.normalize(_vec(clip.get_text_features(**ent)), dim=-1)

# TODO A3: CLIP score = coseno(imagen, texto) × 100
def clip_score(img, texto):
    return None   # ← reemplacen

print(f'control · CLIP score de la imagen del A2 vs su prompt : {clip_score(img_final, PROMPT):.1f}')
print(f'control · CLIP score de los gatos del A1 vs ese prompt: {clip_score(img_a, PROMPT):.1f}')

In [ ]:
GUIDANCES = [1, 3, 7.5, 15]
res_g = []
for g in GUIDANCES:
    img, seg = generar_img(pipe, PROMPT, guidance_scale=g)
    res_g.append((g, img, seg))
mostrar([r[1] for r in res_g], [f'guidance {r[0]}  ·  {r[2]:.1f} s' for r in res_g])

In [ ]:
print(f'{"guidance":>9} | {"CLIP score":>10} | {"segundos":>8}')
print('-' * 35)
for g, img, seg in res_g:
    print(f'{g:>9} | {clip_score(img, PROMPT):>10.1f} | {seg:>8.1f}')

> 👀 **Lo típico:** con 1 la imagen es borrosa o no sigue el prompt; entre 3 y 7.5 la fidelidad sube; hacia 15 el CLIP score se estanca (o hasta baja) y la imagen se **sobresatura**, con colores quemados y artefactos. Más guidance no es mejor: hay un punto dulce, y depende del prompt. Fíjense también en que guidance 1 tarda cerca de la mitad (una pasada de la U-Net en vez de dos).
>
> Todo esto solo se puede comparar porque **la semilla está fija**. Cambien `SEED` y repitan: la imagen cambia entera, la tendencia no. Con una sola semilla la diferencia de CLIP score entre dos guidances puede ser ruido; en M4, promedien sobre varias.

### A4 · ¿Cuántos pasos?

Mismo prompt, misma semilla, guidance 7.5, con 5, 15 y 30 pasos.

In [ ]:
res_p = []
for n in [5, 15, 30]:
    img, seg = generar_img(pipe, PROMPT, guidance_scale=7.5, num_inference_steps=n)
    res_p.append((n, img, seg))
mostrar([r[1] for r in res_p], [f'{r[0]} pasos  ·  {r[2]:.1f} s' for r in res_p])
for n, img, seg in res_p:
    print(f'{n:>3} pasos → {seg:5.1f} s  ·  {seg / n * 1000:4.0f} ms/paso')

> 👀 El tiempo crece casi lineal con los pasos: cada paso es una pasada (o dos, con CFG) de la U-Net. El DDPM original necesitaba **~1000 pasos**; los schedulers modernos (DDIM, DPM-Solver) llegan a buena calidad con **20–30**, y los modelos destilados (celda opcional de SD-Turbo, y la S14) con 1–4.

### De vuelta del Lab A

Anoten para su proyecto (M4):

| | |
|---|---|
| Prompt de su dominio | … |
| Mejor `guidance_scale` (y su CLIP score) | … |
| Pasos que bastan | … |
| Segundos por imagen en T4 (su latencia) | … |

## 3 · Lab B · Control con ControlNet

El prompt controla el **qué**, pero la composición (dónde queda cada cosa) sale de la semilla: es azar. **ControlNet** agrega una copia entrenable del encoder de la U-Net que recibe una imagen de condición (bordes, pose, profundidad…) y le suma su señal a la U-Net original, que queda congelada. Así fijamos la **estructura** y el texto sigue decidiendo el contenido y el estilo.

Usamos el ControlNet de bordes **Canny** para SD 1.5 (~0,7 GB en fp16) y armamos el pipeline **reusando las piezas ya cargadas** (misma U-Net, VAE, text encoder y scheduler): no duplicamos VRAM.

In [ ]:
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline
CN_ID = 'lllyasviel/control_v11p_sd15_canny'
controlnet = ControlNetModel.from_pretrained(CN_ID, variant='fp16', **DT).to(device)

# pipe.components trae vae, text_encoder, tokenizer, unet, scheduler, safety_checker, feature_extractor
# e image_encoder (None). Filtramos por la firma del pipeline nuevo, por si alguna versión cambia las claves.
firma = inspect.signature(StableDiffusionControlNetPipeline.__init__).parameters
piezas = {k: v for k, v in pipe.components.items() if k in firma}
pipe_cn = StableDiffusionControlNetPipeline(**piezas, controlnet=controlnet)
pipe_cn.set_progress_bar_config(leave=False)

print('comparte la U-Net con pipe:', pipe_cn.unet is pipe.unet)
if device == 'cuda':
    print(f'VRAM ocupada: {torch.cuda.memory_allocated() / 1e9:.1f} GB de {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB')

### B1 · Los bordes de una imagen de referencia

**TODO B1:** completen `bordes_canny`: `cv2.Canny(arreglo, bajo, alto)` devuelve una imagen de **un canal** (0 o 255); ControlNet espera **3 canales**. Apílenlo tres veces y conviértanlo a PIL.

> **Ojo:** esta celda falla hasta que completen el TODO (en la versión de clase devuelve `None`).

In [ ]:
# 👉 SU IMAGEN AQUÍ (URL o ruta). Por defecto, la sala de COCO de la S11.
IMG_REF = 'http://images.cocodataset.org/val2017/000000000139.jpg'
ref = cuadrada(cargar(IMG_REF))

# TODO B1: bordes Canny (umbrales 100/200) → PIL de 3 canales
def bordes_canny(img, bajo=100, alto=200):
    return None   # ← reemplacen

bordes = bordes_canny(ref)
mostrar([ref, bordes], ['referencia', 'bordes Canny (100/200)'], tam=4)

> Si los bordes salen muy cargados (texturas, pasto), suban los umbrales (p. ej. 150/250); si salen muy pobres, bájenlos (50/150). Lo que ControlNet recibe es esta imagen, no la foto.

### B2 · Mismo prompt, sin y con ControlNet

In [ ]:
PROMPT_B = 'a cozy scandinavian living room, watercolor painting, soft pastel colors'   # 👉 editable

sin_cn, s1 = generar_img(pipe, PROMPT_B)
con_cn, s2 = generar_img(pipe_cn, PROMPT_B, image=bordes)
mostrar([bordes, sin_cn, con_cn], ['condición', f'sin ControlNet · {s1:.1f} s', f'con ControlNet · {s2:.1f} s'], tam=4)

> 👀 Misma semilla, mismo prompt. **Sin** control, la composición la decide el ruido inicial; **con** control, la imagen respeta los bordes de la referencia (muebles, ventanas, perspectiva) y el prompt pone el estilo. El costo: ControlNet es una red más por paso, así que tarda algo más. Esa latencia también se reporta.

### B3 · ¿Cuánto control? `controlnet_conditioning_scale`

In [ ]:
res_c = []
for esc in [0.3, 0.7, 1.0]:
    img, seg = generar_img(pipe_cn, PROMPT_B, image=bordes, controlnet_conditioning_scale=esc)
    res_c.append((esc, img, seg))
mostrar([r[1] for r in res_c], [f'escala {r[0]}  ·  CLIP {clip_score(r[1], PROMPT_B):.1f}' for r in res_c], tam=3.6)

> 👀 Con 0.3 los bordes son una sugerencia; con 1.0 son casi una orden. Si el prompt pide algo que no cabe en esa estructura (un "bosque" sobre los bordes de una sala), una escala alta pelea con el texto y el CLIP score puede bajar. Otra vez: hay un punto dulce, y se mide.

### B4 · Su dominio

Generen **al menos una** imagen condicionada para SU proyecto: su imagen de referencia (o la de arriba) y su prompt.

👉 SU PROMPT AQUÍ — en inglés, describiendo contenido y estilo (p. ej. *"a product photo of a ceramic mug on a wooden table, studio lighting"*).

In [ ]:
MI_IMAGEN = IMG_REF                                                       # 👉 SU IMAGEN AQUÍ (URL o ruta)
MI_PROMPT = 'a modern coworking space in Medellin, photorealistic, daylight'   # 👉 SU PROMPT AQUÍ
MI_ESCALA = 0.7

mi_ref = cuadrada(cargar(MI_IMAGEN))
mis_bordes = bordes_canny(mi_ref)
mi_img, seg = generar_img(pipe_cn, MI_PROMPT, image=mis_bordes, controlnet_conditioning_scale=MI_ESCALA)
mostrar([mi_ref, mis_bordes, mi_img], ['su referencia', 'bordes', 'generada'], tam=4)
print(f'CLIP score: {clip_score(mi_img, MI_PROMPT):.1f}  ·  {seg:.1f} s  ·  semilla {SEED}')

### De vuelta del Lab B

1. ¿En qué parte de su proyecto tendría sentido **visión como output** (generar una imagen)? ¿O su patrón es input o tool?
2. Si generan imágenes, ¿cómo lo medirían? Piensen en tres capas:
   - **CLIP score** contra el prompt (automático, barato, pero ciego a detalles);
   - un **juez o checklist humano** con criterios de su dominio (¿respeta la estructura?, ¿hay artefactos?, ¿sirve para el usuario?);
   - **latencia** por imagen (los segundos que midieron hoy).
3. ¿Qué dejarían fijo para que las comparaciones sean justas? (semilla, pasos, guidance, escala).

## 4 · Opcional · IP-Adapter: una imagen como prompt

IP-Adapter agrega a la U-Net una segunda cross-attention que mira los rasgos de una **imagen de referencia** (codificada con un encoder de imagen de CLIP): el texto dice qué, la imagen aporta estilo y contenido. Se carga sobre el pipe base.

> Como `pipe` y `pipe_cn` **comparten la U-Net**, al final de la celda descargamos el adaptador (`unload_ip_adapter`) para no afectar el Lab B. Descarga ~45 MB del adaptador + ~2,5 GB del encoder de imagen (ViT-H de OpenCLIP).

In [ ]:
HACER_IP_ADAPTER = False   # ← cámbienlo a True para probar

if HACER_IP_ADAPTER:
    pipe.load_ip_adapter('h94/IP-Adapter', subfolder='models', weight_name='ip-adapter_sd15.safetensors')
    res_ip = []
    for esc in [0.3, 0.7]:
        pipe.set_ip_adapter_scale(esc)
        img, seg = generar_img(pipe, 'a cat sitting in a living room, best quality', ip_adapter_image=ref)
        res_ip.append((esc, img, seg))
    mostrar([ref] + [r[1] for r in res_ip], ['imagen de referencia'] + [f'IP-Adapter {r[0]} · {r[2]:.1f} s' for r in res_ip], tam=3.6)
    pipe.unload_ip_adapter()
    liberar()

## 5 · Opcional · SD-Turbo: una imagen en un paso

`stabilityai/sd-turbo` es un modelo **destilado** (adversarial diffusion distillation): un alumno aprende a hacer en 1 paso lo que el maestro hacía en decenas. Se usa **sin guidance** (`guidance_scale=0.0`). Conecta con la S14 (destilación y eficiencia).

> Descarga ~2,6 GB extra. Tiene licencia propia de Stability AI: si Hugging Face pide aceptarla, entren a la página del modelo con su cuenta y acéptenla (y hagan login con un token en Colab). Por eso es opcional.

In [ ]:
HACER_TURBO = False   # ← cámbienlo a True para probar

if HACER_TURBO:
    from diffusers import AutoPipelineForText2Image
    turbo = AutoPipelineForText2Image.from_pretrained('stabilityai/sd-turbo', variant='fp16', **DT).to(device)
    turbo.set_progress_bar_config(leave=False)
    img_t, seg_t = generar_img(turbo, PROMPT, num_inference_steps=1, guidance_scale=0.0)
    img_b, seg_b = generar_img(pipe, PROMPT, guidance_scale=7.5)
    mostrar([img_b, img_t], [f'SD 1.5 · 25 pasos · {seg_b:.1f} s', f'SD-Turbo · 1 paso · {seg_t:.2f} s'], tam=4)
    print(f'CLIP score · SD 1.5: {clip_score(img_b, PROMPT):.1f}  ·  SD-Turbo: {clip_score(img_t, PROMPT):.1f}')
    del turbo; liberar()

## Lo que se llevan

- **Difusión = aprender a quitar ruido.** El forward ensucia con una fórmula; la red aprende a predecir el ruido, y generar es quitarlo paso a paso desde ruido puro.
- **Guidance = cuánto obedecer al texto.** Sube la fidelidad hasta un punto; más allá, sobresatura. Se elige midiendo, no a ojo.
- **Stable Diffusion trabaja en un espacio latente pequeño** (el VAE comprime 48×): por eso cabe en una T4.
- **ControlNet controla la estructura**; el prompt, el contenido y el estilo.
- Para **M4** (S13 · 15 OCT · cierra dom 25 OCT): si su patrón es **output**, la imagen generada se mide — **CLIP score + juez/checklist + latencia** — y **la semilla se fija** para que las comparaciones sean justas.

*SI4006 · Universidad EAFIT · Sesión 12.*